# Here, we will demonstrate Reflexion

We use the Solver that we created earlier and instead of using the Rejector, we use the Reflector and show the steps of Reflexion.

In [4]:
from prompt_template import Reflector, Solver
from multi_agent import Problem, conversation
import importlib
import reflexion
importlib.reload(reflexion)

q = """
Define $\\operatorname{sgn}(x)$ to be $1$ when $x$ is positive, $-1$ when $x$ is $0$.
Compute $$ \\sum_{n=1}^{\\infty} \\frac{\\operatorname{sgn}\\left(\\sin\\left(2^{n}\\right)\\right)}{2^{n}} $$
(The arguments to sin are in radians.)
"""

a = "answer 1-\\frac{2}{\\pi}"

problem = problem = Problem(
    roles=[Solver, Reflector],  # add Solver if you have one
    problem_descr=q,
    answer=a
)

print(problem)

Welcome Solver, and Reflector. Together, you should solve the following problem:
>>  Define $\operatorname{sgn}(x)$ to be $1$ when $x$ is positive, $-1$ when $x$
is $0$. Compute $$ \sum_{n=1}^{\infty}
\frac{\operatorname{sgn}\left(\sin\left(2^{n}\right)\right)}{2^{n}} $$ (The
arguments to sin are in radians.) .<<  "When you are done, you should submidt
your answer as: ANSWER: <your answer>.  No latex formatting, just the raw
number/numbers or strings at the very end.  Before you start sharing your
toughts, give a little summary of the conversation so far.  Give a list of the
currently suggested answers. Everytime you propose an aswer, check this list.
You proposal cannot be in this this list. Try again and submit a new unique
answer."


In [5]:
from azure_api import Client 
api_version="2024-06-01"
model_name="DeepSeek-R1"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

def evaluator_fn(task, attempt):
    normalized_attempt = attempt.replace(" ", "").replace("\\", "")
    normalized_answer = a.replace(" ", "").replace("\\", "")
    if normalized_answer in normalized_attempt:
        return True, "Correct"
    return False, "Incorrect, needs improvement"


In [6]:
from reflexion import ReflexionAgent, ReflexionStrategy

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=evaluator_fn,
    reflector_prompt=Reflector,
    max_attempts=3
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)


=== Attempt 1 ===


To compute the sum \(\sum_{n=1}^{\infty} \frac{\operatorname{sgn}\left(\sin\left(2^{n}\right)\right)}{2^{n}}\), where \(\operatorname{sgn}(x)\) is defined as 1 if \(x\) is positive and -1 if \(x\) is zero or negative, we need to determine the sign of \(\sin(2^n)\) for each \(n\).

1. **Understanding the Sign Function**: The problem defines \(\operatorname{sgn}(x)\) as 1 for positive \(x\) and -1 for non-positive \(x\). This means \(\operatorname{sgn}(\sin(2^n))\) is 1 if \(\sin(2^n) > 0\) and -1 otherwise.

2. **Modulo Operation**: The value of \(2^n \mod 2\pi\) determines the angle in the interval \([0, 2\pi)\). The sine of this angle will be positive if the angle is in \((0, \pi)\) and negative if it is in \((\pi, 2\pi)\).

3. **Binary Expansion Connection**: The sequence \(2^n \mod 2\pi\) corresponds to the fractional part of \(2^{n-1}/\pi\). This is related to the binary expansion of \(1/\pi\). The sign of \(\sin(2^n)\) is determined by the binary digits of thi